# IOAI — 2026 Home Task Robot Delivery (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
os.makedirs('data', exist_ok=True)
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-home-task-robot-delivery'
for f in ['train_demos.pkl','test_scenarios.pkl']:
    if not os.path.exists(f'data/{f}'): os.system(f'wget -q {BASE}/{f} -O data/{f}')
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Robot Delivery — 모범답안 (BFS 경로계획)

모방학습 대신 **정확한 경로계획**으로 푼다. 격자는 완전관측이므로 BFS 로 (agent→소포 depot)과
(소포→목적지 depot) 최단경로를 구하고 사이에 pickup/dropoff 를 끼워 넣으면 **항상 성공**한다 → 성공률 1.0.
(원 대회는 은닉 테스트라 학습모델을 요구하지만, 여기서는 결정적 계획으로 상한을 보여준다.)

In [ ]:
import json, pickle
import numpy as np
GRID_SIZE, N_DEPOTS, MAX_STEPS = 8, 6, 120
ACTION_NAMES = {0:"south",1:"north",2:"east",3:"west",4:"pickup",5:"dropoff"}
ACTION_DELTAS = {0:(1,0),1:(-1,0),2:(0,1),3:(0,-1)}

class DeliverySimulator8x8:
    """8x8 배달 시뮬레이터 (IOAI-2026 Home Task 2 원본과 동일)."""
    def reset(self, sc):
        self.step_count=0; self.carrying=False
        self.walls={tuple(c) for c in sc["walls"]}; self.depots=[tuple(c) for c in sc["depots"]]
        self.agent_pos=tuple(sc["agent_pos"]); self.package_location=int(sc["package_location"]); self.destination=int(sc["destination"])
        return self.state()
    def state(self):
        pf = N_DEPOTS if self.carrying else self.package_location
        return int(self.agent_pos[0]), int(self.agent_pos[1]), int(pf), int(self.destination)
    def can_enter(self, r, c):
        return 0<=r<GRID_SIZE and 0<=c<GRID_SIZE and (r,c) not in self.walls
    def valid_action_mask(self):
        r,c,_,dest=self.state(); mask=np.zeros(6,bool)
        for a,(dr,dc) in ACTION_DELTAS.items(): mask[a]=self.can_enter(r+dr,c+dc)
        mask[4]=(not self.carrying) and self.agent_pos==self.depots[self.package_location]
        mask[5]=self.carrying and self.agent_pos==self.depots[dest]
        return mask
    def observation(self):
        row,col,pf,dest=self.state(); carrying=pf==N_DEPOTS
        dest_row,dest_col=self.depots[dest]
        target=(dest_row,dest_col) if carrying else self.depots[pf]; tr,tc=target
        grid=np.zeros((6,GRID_SIZE,GRID_SIZE),np.float32)
        for wr,wc in self.walls: grid[0,wr,wc]=1.0
        for dr,dc in self.depots: grid[1,dr,dc]=1.0
        grid[2,row,col]=1.0
        if not carrying: pr,pc=self.depots[pf]; grid[3,pr,pc]=1.0
        grid[4,dest_row,dest_col]=1.0; grid[5,:,:]=float(carrying)
        blocked=[float(not self.can_enter(row+dr,col+dc)) for dr,dc in ACTION_DELTAS.values()]
        vector=np.array([row/(GRID_SIZE-1),col/(GRID_SIZE-1),pf/N_DEPOTS,dest/(N_DEPOTS-1),float(carrying),
                         tr/(GRID_SIZE-1),tc/(GRID_SIZE-1),(tr-row)/(GRID_SIZE-1),(tc-col)/(GRID_SIZE-1),*blocked],np.float32)
        return {"grid":grid,"vector":vector,"action_mask":self.valid_action_mask(),"state":self.state()}
    def step(self, action):
        action=int(action); done=False
        if action in ACTION_DELTAS:
            dr,dc=ACTION_DELTAS[action]; r,c=self.agent_pos[0]+dr,self.agent_pos[1]+dc
            if self.can_enter(r,c): self.agent_pos=(r,c)
        elif action==4 and (not self.carrying) and self.agent_pos==self.depots[self.package_location]: self.carrying=True
        elif action==5 and self.carrying and self.agent_pos==self.depots[self.destination]: done=True; self.carrying=False
        elif action not in (4,5): raise ValueError(f"unknown action {action}")
        self.step_count+=1
        return self.state(), done, (self.step_count>=MAX_STEPS and not done), {}

def flatten_observation(obs):
    return np.concatenate([obs["grid"].reshape(-1), obs["vector"]]).astype(np.float32)

def run_episode(scenario, action_fn):
    sim=DeliverySimulator8x8(); sim.reset(scenario); actions=[]
    for _ in range(MAX_STEPS):
        a=int(action_fn(sim.observation())); _,done,timed,_=sim.step(a); actions.append(a)
        if done or timed: break
    return {"success": done if 'done' in dir() else False, "actions": actions}

def generate_predictions(scenarios, action_fn, path="predictions.jsonl"):
    with open(path,"w") as f:
        for sc in scenarios:
            ep=run_episode(sc, action_fn)
            f.write(json.dumps({"layout_id":sc["layout_id"],"episode_seed":sc["episode_seed"],"actions":ep["actions"]})+"\n")
    print("saved", path, len(scenarios), "scenarios")

In [ ]:
train = pickle.load(open("data/train_demos.pkl","rb"))["trajectories"]   # 400 expert demos
test_scenarios = pickle.load(open("data/test_scenarios.pkl","rb"))       # 200 to solve
print("demos:", len(train), "| test scenarios:", len(test_scenarios))

In [ ]:
from collections import deque
def bfs(start, goal, walls):
    if start==goal: return []
    q=deque([(start,[])]); seen={start}
    while q:
        pos,path=q.popleft()
        for a,(dr,dc) in ACTION_DELTAS.items():
            nr,nc=pos[0]+dr,pos[1]+dc; npos=(nr,nc)
            if 0<=nr<GRID_SIZE and 0<=nc<GRID_SIZE and npos not in walls and npos not in seen:
                if npos==goal: return path+[a]
                seen.add(npos); q.append((npos,path+[a]))
    return None

def plan_action_seq(sc):
    walls={tuple(c) for c in sc["walls"]}; depots=[tuple(c) for c in sc["depots"]]
    agent=tuple(sc["agent_pos"]); pkg=depots[sc["package_location"]]; dest=depots[sc["destination"]]
    p1=bfs(agent,pkg,walls); p2=bfs(pkg,dest,walls)
    if p1 is None or p2 is None: return []
    return p1+[4]+p2+[5]

# 계획된 행동열을 그대로 제출(시뮬레이터가 재생)
with open("predictions.jsonl","w") as f:
    for sc in test_scenarios:
        f.write(json.dumps({"layout_id":sc["layout_id"],"episode_seed":sc["episode_seed"],"actions":plan_action_seq(sc)})+"\n")
print("saved predictions.jsonl (BFS)", len(test_scenarios))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['predictions.jsonl']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)